In [1]:
from run_main import run
import json
import pandas as pd
from IPython.display import Image, JSON, display
import helper_funcs as hf

Start with the worm model optimized for forward locomotion:

In [2]:
indir = 'example_W2D18L'
outdir = indir + "_CO2out"
pfolder = 'notebooks'
inputFolderName = pfolder + "/" + indir
outputFolderName = pfolder + "/" + outdir

Delete the output folder if necessary

In [3]:
hf.delete_subfolder_directory(pfolder, outdir)

True

In [4]:
json_data = hf.get_worm_json(inputFolderName)

Add a simple linearly decreasing concentration environment, with peak at the origin and gradient 0.5. The `environments` object defines one or more spatial chemical environments that sensors can monitor. Each environment has a unique `name`, a source location given by `x_center` and `y_center`, and a `grad_steep` value controlling how quickly concentration changes with distance. Sensors select an environment by referencing its name. Multiple sensors may share the same environment.

In [5]:
json_data=hf.add_environment(json_data, "salt_environment")
print('environments: ' + json.dumps(json_data['environments'], indent=2))

environments: {
  "salt_environment": {
    "name": {
      "value": "salt_environment"
    },
    "x_center": {
      "value": 0.0
    },
    "y_center": {
      "value": 0.0
    },
    "grad_steep": {
      "value": 0.5
    }
  }
}


Add a simple sensor of this environment. The sensor detects whether environmental concentration at the worm’s head is increasing or decreasing. It compares a recent average over `sensor_n` seconds with an earlier average over `sensor_m` seconds. `output_1` represents increasing concentration, while `output_2` represents decreasing concentration. The `weights` connect these outputs to nervous-system cells, and `environment` selects the sensed environment.

In [6]:
json_data=hf.add_sensor(json_data, "salt_environment")
print('sensors: ' + json.dumps(json_data['sensors'], indent=2))

sensors: {
  "sensor_1": {
    "environment": {
      "value": "salt_environment"
    },
    "sensor_m": {
      "value": 2.0
    },
    "sensor_n": {
      "value": 2.0
    },
    "outputs": {
      "message": "Available sensor output names for sensor-to-cell connections",
      "value": [
        {
          "name": "output_1",
          "description": "Positive change in sensed concentration (present average above past average)"
        },
        {
          "name": "output_2",
          "description": "Negative change in sensed concentration (past average above present average)"
        }
      ]
    },
    "weights": {
      "message": "Weights from sensor outputs to Nervous System cells",
      "value": []
    }
  }
}


Next we add a random network. The `add_random_cell_network` function adds a specified number of new interneurons to the JSON nervous system and returns the updated JSON together with their names. Cells are named `Cell_1`, `Cell_2`, and so on, skipping names already in use. Every directed pair of new cells is given a chemical connection with the requested probability, with weights sampled uniformly between `-1` and `1`; reciprocal connections may occur independently. The optional `random_seed` makes the generated network reproducible. The new network is initially isolated from all pre-existing cells.

In [8]:
json_data, cell_names=hf.add_random_cell_network(json_data, 4, 1)
print(cell_names)
print(json.dumps(json_data['nervous_system']['cells']['DB_0'], indent=2))